# Deliverable 2 -- Prompting Mejorado con Few-Shot y Post-procesamiento

In [ ]:
import json
import re
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODELOS = {
    "tinyllama": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "qwen": "Qwen/Qwen2-0.5B-Instruct",
}

MODELO_ACTUAL = "tinyllama"  # Cambiar a "qwen" para evaluar Qwen
MODELO_ID = MODELOS[MODELO_ACTUAL]
FECHA_REFERENCIA = "2026-09-01"
DIA_REFERENCIA = "martes"

print(f"Modelo: {MODELO_ID}")
print(f"Fecha de referencia: {DIA_REFERENCIA} {FECHA_REFERENCIA}")

In [ ]:
# ============================================================
# CARGA O GENERACION DEL BENCHMARK
# ============================================================

BENCHMARK_FILE = "frases.json"

if os.path.exists(BENCHMARK_FILE):
    with open(BENCHMARK_FILE, "r", encoding="utf-8") as f:
        benchmark = json.load(f)
    print(f"Benchmark cargado: {len(benchmark['frases'])} frases")
else:
    print("Generando benchmark embebido...")
    benchmark = {
        "metadata": {"fecha_referencia": "2026-09-01", "dia_referencia": "martes", "total_frases": 35},
        "frases": [
            {"id": 1, "categoria": "simple", "texto": "Reuni\u00f3n el lunes a las 3pm", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-07", "hora_inicio": "15:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 2, "categoria": "simple", "texto": "Cita el martes 2 de septiembre a las 14:00", "ground_truth": {"titulo": "Cita", "fecha": "2026-09-02", "hora_inicio": "14:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 3, "categoria": "simple", "texto": "Reuni\u00f3n el 15 de octubre a las 10 de la ma\u00f1ana", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-10-15", "hora_inicio": "10:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 4, "categoria": "simple", "texto": "Meeting on Friday at 10am", "ground_truth": {"titulo": "Meeting", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 5, "categoria": "simple", "texto": "Reuni\u00f3n el jueves 10/09 a las 8:30", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-10", "hora_inicio": "08:30", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 6, "categoria": "simple", "texto": "Reuni\u00f3n el viernes a las 5pm", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-04", "hora_inicio": "17:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 7, "categoria": "simple", "texto": "Reuni\u00f3n el martes 23 de noviembre a las 16:00", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-11-23", "hora_inicio": "16:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 8, "categoria": "simple", "texto": "Appointment on December 5th at 2pm", "ground_truth": {"titulo": "Appointment", "fecha": "2026-12-05", "hora_inicio": "14:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 9, "categoria": "con_participantes", "texto": "Reuni\u00f3n con Ana y Pedro el viernes a las 10:00", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": None, "participantes": ["Ana", "Pedro"], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 10, "categoria": "con_participantes", "texto": "Reuni\u00f3n de equipo el mi\u00e9rcoles 15/09 a las 9:30 con Juan, Mar\u00eda y Carlos", "ground_truth": {"titulo": "Reuni\u00f3n de equipo", "fecha": "2026-09-15", "hora_inicio": "09:30", "duracion_minutos": None, "participantes": ["Juan", "Mar\u00eda", "Carlos"], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 11, "categoria": "con_participantes", "texto": "Reuni\u00f3n con el cliente el lunes a las 11am, participan: Roberto, Laura", "ground_truth": {"titulo": "Reuni\u00f3n con el cliente", "fecha": "2026-09-07", "hora_inicio": "11:00", "duracion_minutos": None, "participantes": ["Roberto", "Laura"], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 12, "categoria": "con_participantes", "texto": "Reuni\u00f3n con Sof\u00eda el martes 8 de septiembre a las 15:30", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-08", "hora_inicio": "15:30", "duracion_minutos": None, "participantes": ["Sof\u00eda"], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 13, "categoria": "con_participantes", "texto": "Reuni\u00f3n de trabajo con Diego, Valentina y Andr\u00e9s el jueves a las 14:00", "ground_truth": {"titulo": "Reuni\u00f3n de trabajo", "fecha": "2026-09-03", "hora_inicio": "14:00", "duracion_minutos": None, "participantes": ["Diego", "Valentina", "Andr\u00e9s"], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 14, "categoria": "con_participantes", "texto": "Call with John and Maria on Wednesday at 3pm", "ground_truth": {"titulo": "Call", "fecha": "2026-09-02", "hora_inicio": "15:00", "duracion_minutos": None, "participantes": ["John", "Maria"], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 15, "categoria": "con_participantes", "texto": "Reuni\u00f3n con el equipo de desarrollo el lunes 14/09 a las 9:00", "ground_truth": {"titulo": "Reuni\u00f3n con el equipo de desarrollo", "fecha": "2026-09-14", "hora_inicio": "09:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 16, "categoria": "ambigua", "texto": "Reuni\u00f3n la pr\u00f3xima semana por la ma\u00f1ana", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-07", "hora_inicio": "09:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 17, "categoria": "ambigua", "texto": "Reuni\u00f3n ma\u00f1ana a la tarde", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-02", "hora_inicio": "15:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 18, "categoria": "ambigua", "texto": "Reuni\u00f3n el mes que viene", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-10-01", "hora_inicio": None, "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 19, "categoria": "ambigua", "texto": "Reuni\u00f3n en un rato", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-01", "hora_inicio": None, "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 20, "categoria": "ambigua", "texto": "Reuni\u00f3n el finde", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-05", "hora_inicio": None, "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 21, "categoria": "ambigua", "texto": "Reuni\u00f3n a mediados de septiembre", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-15", "hora_inicio": None, "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 22, "categoria": "ambigua", "texto": "Reuni\u00f3n el lunes que viene a las 10", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-07", "hora_inicio": "10:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 23, "categoria": "compleja", "texto": "Reuni\u00f3n de 2 horas el 15/09 a las 9:30 con el equipo de ventas en la sala de conferencias, recordar 30 minutos antes", "ground_truth": {"titulo": "Reuni\u00f3n de equipo de ventas", "fecha": "2026-09-15", "hora_inicio": "09:30", "duracion_minutos": 120, "participantes": [], "ubicacion": "Sala de conferencias", "recordatorio_minutos": 30, "zona_horaria": "America/Santiago"}},
            {"id": 24, "categoria": "compleja", "texto": "Reuni\u00f3n con el cliente de Bankard el jueves a las 11am, duraci\u00f3n 1 hora, Zoom link: zoom.us/j/123456, zona horaria Argentina", "ground_truth": {"titulo": "Reuni\u00f3n con cliente de Bankard", "fecha": "2026-09-03", "hora_inicio": "11:00", "duracion_minutos": 60, "participantes": [], "ubicacion": "zoom.us/j/123456", "recordatorio_minutos": None, "zona_horaria": "America/Argentina/Buenos_Aires"}},
            {"id": 25, "categoria": "compleja", "texto": "Reuni\u00f3n de standup diario a las 9:15am, 15 minutos, con el equipo de ingenier\u00eda en Google Meet", "ground_truth": {"titulo": "Reuni\u00f3n de standup diario", "fecha": None, "hora_inicio": "09:15", "duracion_minutos": 15, "participantes": [], "ubicacion": "Google Meet", "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 26, "categoria": "compleja", "texto": "Reuni\u00f3n con Ana, Pedro y Mar\u00eda el martes 8/09 de 14:00 a 16:00 en oficina 501, traer presentaci\u00f3n, recordar 1 hora antes", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-08", "hora_inicio": "14:00", "duracion_minutos": 120, "participantes": ["Ana", "Pedro", "Mar\u00eda"], "ubicacion": "Oficina 501", "recordatorio_minutos": 60, "zona_horaria": "America/Santiago"}},
            {"id": 27, "categoria": "compleja", "texto": "Weekly sync with design team, every Wednesday at 2pm, 45 minutes, Zoom", "ground_truth": {"titulo": "Weekly sync with design team", "fecha": None, "hora_inicio": "14:00", "duracion_minutos": 45, "participantes": [], "ubicacion": "Zoom", "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 28, "categoria": "compleja", "texto": "Reuni\u00f3n de revisi\u00f3n de c\u00f3digo el viernes 11/09 a las 10:00 con el equipo backend, duraci\u00f3n 30 min, sala B2", "ground_truth": {"titulo": "Reuni\u00f3n de revisi\u00f3n de c\u00f3digo", "fecha": "2026-09-11", "hora_inicio": "10:00", "duracion_minutos": 30, "participantes": [], "ubicacion": "Sala B2", "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 29, "categoria": "edge_case", "texto": "Reuni\u00f3n", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": None, "hora_inicio": None, "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 30, "categoria": "edge_case", "texto": "Reuni\u00f3n el 30 de febrero a las 25:00", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": None, "hora_inicio": None, "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 31, "categoria": "edge_case", "texto": "Reuni\u00f3n con 12345 el lunes a las 3pm", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-07", "hora_inicio": "15:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 32, "categoria": "edge_case", "texto": "Reuni\u00f3n el lunes a las 3pm y tambi\u00e9n reuni\u00f3n el martes a las 10am", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-07", "hora_inicio": "15:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 33, "categoria": "edge_case", "texto": "Agenda una reuni\u00f3n para ayer a las 10am", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-08-31", "hora_inicio": "10:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}},
            {"id": 34, "categoria": "edge_case", "texto": "Reuni\u00f3n con Ana Garc\u00eda-P\u00e9rez y Jos\u00e9 Mar\u00eda L\u00f3pez el mi\u00e9rcoles a las 11:30am por 45 minutos en la sala de juntas, recordar 15 minutos antes", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-02", "hora_inicio": "11:30", "duracion_minutos": 45, "participantes": ["Ana Garc\u00eda-P\u00e9rez", "Jos\u00e9 Mar\u00eda L\u00f3pez"], "ubicacion": "Sala de juntas", "recordatorio_minutos": 15, "zona_horaria": "America/Santiago"}},
            {"id": 35, "categoria": "edge_case", "texto": "Reuni\u00f3n el pr\u00f3ximo martes a las 3pm con el equipo, duraci\u00f3n aproximada de 1 a 2 horas", "ground_truth": {"titulo": "Reuni\u00f3n", "fecha": "2026-09-08", "hora_inicio": "15:00", "duracion_minutos": None, "participantes": [], "ubicacion": None, "recordatorio_minutos": None, "zona_horaria": "America/Santiago"}}
        ]
    }
    with open(BENCHMARK_FILE, "w", encoding="utf-8") as f:
        json.dump(benchmark, f, indent=2, ensure_ascii=False)
    print(f"Benchmark auto-generado: {len(benchmark['frases'])} frases")

In [ ]:
# ============================================================
# CARGA DEL MODELO
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID, torch_dtype=torch.float16, device_map="auto"
)
print(f"Modelo cargado en: {model.device}")

In [ ]:
# ============================================================
# SYSTEM PROMPT: Reglas + Few-Shot
# ============================================================

SYSTEM_PROMPT = """Eres un extractor de datos de calendario.

REGLAS:
1. HOY ES {dia} {fecha}. Calcula fechas relativas desde esta fecha.
2. Si un campo NO se menciona, usa null (o [] para participantes). NUNCA inventes.
3. titulo: solo la accion ("Reuni\u00f3n", "Cita", "Meeting", "Call"). Sin fechas ni horas.
4. fecha: YYYY-MM-DD. hora_inicio: HH:MM 24h.
5. duracion_minutos: entero (1 hora=60). Si no hay = null.
6. participantes: lista de nombres reales. Si no hay = [].
7. ubicacion: donde se realiza. Si no se menciona = null.
8. recordatorio_minutos: entero. Si no se menciona = null.
9. zona_horaria: "America/Santiago" salvo que se especifique otra.

EJEMPLOS:

Frase: "Reuni\u00f3n con Ana el viernes a las 10am"
{"titulo": "Reuni\u00f3n", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": null, "participantes": ["Ana"], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}

Frase: "Reuni\u00f3n de 2 horas el 15/09 a las 9:30 en la sala de conferencias, recordar 30 min"
{"titulo": "Reuni\u00f3n", "fecha": "2026-09-15", "hora_inicio": "09:30", "duracion_minutos": 120, "participantes": [], "ubicacion": "Sala de conferencias", "recordatorio_minutos": 30, "zona_horaria": "America/Santiago"}

Frase: "Reuni\u00f3n"
{"titulo": "Reuni\u00f3n", "fecha": null, "hora_inicio": null, "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}

Frase: "Meeting on Friday at 10am"
{"titulo": "Meeting", "fecha": "2026-09-04", "hora_inicio": "10:00", "duracion_minutos": null, "participantes": [], "ubicacion": null, "recordatorio_minutos": null, "zona_horaria": "America/Santiago"}"""

def construir_prompt(frase_texto):
    system = SYSTEM_PROMPT.format(dia=DIA_REFERENCIA, fecha=FECHA_REFERENCIA)
    user_msg = f'Frase: "{frase_texto}"\n\nRespuesta (SOLO JSON):'
    if MODELO_ACTUAL == "qwen":
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user_msg}
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        return f"<|system|>\n{system}\n<|user|>\n{user_msg}\n<|assistant|>\n"

print("Prompt configurado")

In [ ]:
# ============================================================
# POST-PROCESAMIENTO
# ============================================================

def post_procesar(json_salida, frase_texto):
    if not isinstance(json_salida, dict) or "error" in json_salida:
        return json_salida
    r = json_salida.copy()

    # Corregir anho
    f = r.get("fecha")
    if isinstance(f, str):
        m = re.match(r"(\d{4})", f)
        if m and int(m.group(1)) not in (2026, 2027):
            r["fecha"] = f.replace(m.group(1), "2026", 1)

    # Corregir hora
    h = r.get("hora_inicio")
    if isinstance(h, str):
        hm = re.search(r"(\d{1,2}):(\d{2})", h)
        if hm:
            r["hora_inicio"] = f"{int(hm.group(1)):02d}:{int(hm.group(2)):02d}"

    # Limpiar participantes
    p = r.get("participantes", [])
    if isinstance(p, list):
        r["participantes"] = [x for x in p if isinstance(x, str) and not x.startswith("<") and len(x.strip()) > 0 and not x.isdigit()]

    # Numerico
    for campo in ["duracion_minutos", "recordatorio_minutos"]:
        val = r.get(campo)
        if val is not None and not isinstance(val, (int, float)):
            nm = re.search(r"(\d+)", str(val))
            r[campo] = int(nm.group(1)) if nm else None

    # Titulo limpio
    t = r.get("titulo", "")
    if isinstance(t, str) and (len(t) > 50 or re.search(r"\d{1,2}[/-]\d{1,2}", t)):
        for palabra in ["Reuni\u00f3n", "Cita", "Meeting", "Call", "Appointment"]:
            if palabra.lower() in t.lower():
                r["titulo"] = palabra
                break
    return r

print("Post-procesador listo")

In [ ]:
# ============================================================
# EJECUCION
# ============================================================
resultados = []

for frase in benchmark["frases"]:
    prompt = construir_prompt(frase["texto"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=256, temperature=0.01,
            do_sample=True, top_p=0.95, pad_token_id=tokenizer.eos_token_id
        )

    respuesta = tokenizer.decode(outputs[0][len(inputs.input_ids[0]):], skip_special_tokens=True).strip()
    json_match = re.search(r"\{.*\}", respuesta, re.DOTALL)
    json_salida = {"error": "No JSON detected"}
    if json_match:
        try:
            json_salida = json.loads(json_match.group())
        except json.JSONDecodeError:
            json_salida = {"error": "Invalid JSON", "raw": respuesta[:200]}

    json_salida = post_procesar(json_salida, frase["texto"])
    resultados.append({"id": frase["id"], "texto": frase["texto"], "resultado_modelo": json_salida})
    print(f"Frase {frase['id']:2d}: {'OK' if 'error' not in json_salida else 'ERROR'}")

output_file = f"resultados_{MODELO_ACTUAL}_mejorado.json"
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)
print(f"\nGuardado: {output_file}")

In [ ]:
# ============================================================
# EVALUACION vs GROUND TRUTH
# ============================================================
CAMPOS = ["titulo", "fecha", "hora_inicio", "duracion_minutos",
          "participantes", "ubicacion", "recordatorio_minutos", "zona_horaria"]
correctos = 0
errores_campo = {c: 0 for c in CAMPOS}

for r in resultados:
    if "error" in r["resultado_modelo"]:
        continue
    gt = next(f for f in benchmark["frases"] if f["id"] == r["id"])["ground_truth"]
    pred = r["resultado_modelo"]
    ok = True
    for c in CAMPOS:
        esp, opt = gt.get(c), pred.get(c)
        if esp is None and opt is None: continue
        elif esp is None and opt is not None: errores_campo[c] += 1; ok = False
        elif esp is not None and opt is None: errores_campo[c] += 1; ok = False
        elif isinstance(esp, list):
            if set(str(x).lower() for x in esp) != set(str(x).lower() for x in (opt or [])):
                errores_campo[c] += 1; ok = False
        elif str(esp).lower() != str(opt).lower():
            errores_campo[c] += 1; ok = False
    if ok: correctos += 1

total = len(resultados)
print(f"\n{'='*50}")
print(f"  {MODELO_ACTUAL.upper()} MEJORADO")
print(f"  Precision: {correctos}/{total} ({correctos/total*100:.1f}%)")
print(f"{'='*50}")
for c in CAMPOS:
    ok = total - errores_campo[c]
    print(f"  {c:25s}: {ok}/{total} ({ok/total*100:.1f}%)")